In [84]:
import pandas as pd
import numpy as np
from config_ import *



Galeras Cleaning

base_grandes =

rasgos =

#unir estas dos primero y tener unificado, asegurandonos que estamos bien
#unir con cellulosa y bulk



In [85]:
base_grande = pd.read_excel(r"\Users\selene.baez\Downloads\NAPO_preliminar\Galeras\Base_Galeras_vK.xlsx")
rasgos = pd.read_excel(r"C:\Users\selene.baez\Downloads\NAPO_preliminar\Galeras\03. Rasgos Galeras.xlsx")
#en rasgos solo mantener las columnas de los rasgos. 

In [86]:
rasgos = rasgos.rename(columns={"#":"Plot", "TreeID_2025":"TreeID"}) #Nunca usar Familia y genus de este.
rasgos.head(3)
#possible errores por nombres antiguos. Reeplace first usando la del 2025 en las bases.

,Plot,Site,Plot_code,Sub,Date,TreeID,Family,genus,specie,altitude,...,Leaf dry weight (g),Leaf area (cm²),SLA,Unnamed: 25,Comment,wood characteristics,crust thickness,wet lenght cm,wet wood weight,dry wood weight
0,24,Galeras,GAL_24,A,2025-01-08,8633,Clusiaceae,Garcinia,madruno,1450,...,17.47,1568.870,89.803663,NaN,NaN,NaN,0.1,6.4,1.66,0.86
1,24,Galeras,GAL_24,B,2025-01-08,8639,Lacistemataceae,Lozania,klugii,1450,...,10.22,1594.462,156.013894,NaN,NaN,NaN,0.3,8.1,2.02,0.84
2,24,Galeras,GAL_24,B,2025-01-08,8640,indet,NaN,NaN,1450,...,9.03,645.064,71.435659,NaN,mas 10 Nutrientes,NaN,0.2,5.4,1.25,0.42


In [87]:
merge = pd.merge(base_grande, rasgos, how="outer", left_on = ["new ID 2024","Plot"],right_on= ["TreeID","Plot"])
merge = merge[merge["PlotID"].isin([
    "GAL_22", "GAL_23", "GAL_24", "GAL_28",
    "GAL_48", "GAL_50", "GAL_60", "GAL_62", "GAL_29"
])] #filtra los censados. GAL 29 no hecha pero hay del 2006

#merge.to_excel("testGaleras.xlsx")

In [88]:
#CELLULOSA
celulosa = pd.read_excel(r"C:\Users\selene.baez\Downloads\NAPO_preliminar\Galeras\resultados072025.xlsx")
patrones = ["p60", "p62", "p22","p23","p24","p28","p48","p50","p29"]
df_filtrado = celulosa[celulosa["Sample ID"].str.contains("|".join(patrones), na=False)]
## arreglando typos
df_filtrado["Sample ID"] = df_filtrado["Sample ID"].replace({
    "p28-8974": "p23-8974",
    "p28-8982": "p23-8982",
    "p28-8976": "p23-8976"})

df_filtrado["Year"] = np.select(
    [
        df_filtrado["Meas batch"].str.contains("NAPO2006", na=False),
        df_filtrado["Meas batch"].str.contains("NUMEX", na=False), #caso especial
        df_filtrado["Meas batch"].str.contains("NAPO2025", na=False),
    ],
    [2006,2006,2025],
    default=np.nan
)

df_filtrado["Site"] = "Galeras"
#HELP: La idea es filtrar y luego de filtrar por año se puede unir por ID 2025 o ID original
df_filtrado[["Plot","TreeID"]] = df_filtrado["Sample ID"].str.split("-",expand=True)
df_filtrado["Type"] = "leaf pulverized"
df_filtrado["Plot"]= df_filtrado["Plot"].str.replace("p","")
df_filtrado = df_filtrado.rename(columns = {"δ15N (‰ v.s. AIR)":named13C_cellulose})

print("Resultados de Celulosa")
len(df_filtrado) #78


Resultados de Celulosa


C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\227471084.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["Sample ID"] = df_filtrado["Sample ID"].replace({
C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\227471084.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtrado["Year"] = np.select(
C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\227471084.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

78

# Celulosa de muestras de herbario y muestras enero 2026.

In [89]:
cel2026 = pd.read_excel(r"C:\Users\selene.baez\Downloads\NAPO_preliminar\Galeras\Listado_celulosa_2026.xlsx")
cel2026["Sample-ID"] = cel2026["Sample-ID"].replace({"3872":"jh-3872","3896":"jh-3896","3885":"jh-3885","jh3697":"jh-3697"})

cel2026[["Plot","numeroCampo"]] = cel2026["Sample-ID"].str.split("-",expand=True) #aplica para los que sean herbario
cel2026["Plot"] = cel2026["Plot"].str.replace("p","")

cel2026.loc[cel2026["Plot"] != "jh", "Type"] = "leaf pulverized"
cel2026.loc[cel2026["Plot"] != "jh", "Year"] = 2025
cel2026.loc[cel2026["Plot"] == "jh", "Type"] = "leaf herbarium"
cel2026.loc[cel2026["Plot"] == "jh", "Year"] = 2006
cel2026["numeroCampo"] = cel2026["numeroCampo"].astype("float64")
cel2026[nameC_cellulose] = "PENDING"
cel2026[named13C_cellulose] = "PENDING"
cel2026[named13C_bulk] = "PENDING"
cel2026[nameC_bulk] = "PENDING"
cel2026[named15N] = "PENDING"


In [ ]:
### Cleaning of the new cellulose results. 
### work only with pulverized samples ####
pulv2026 = cel2026[cel2026["Type"] == "leaf pulverized"] ##99
pulv2026 [["Plot","new TreeID"]] = pulv2026["Sample-ID"].str.split("-",expand=True)
pulv2026["Plot"]= pulv2026["Plot"].str.replace("p","") #homogeneizar
patrones = ["60", "62", "22","23","24","28","48","50","29"]
pulv2026Gal = pulv2026[pulv2026["Plot"].str.contains("|".join(patrones), na=False)]
pulv2026Gal["Site"] = "Galeras"
pulv2026Gal["new TreeID"] = pulv2026Gal["new TreeID"].astype("float64")
pulv2026Complete = pd.merge(pulv2026Gal,base_grande, left_on= "new TreeID",right_on="new ID 2024",how="left")
### only working with samples 2006 jh herbarium ##
herb_2006 = cel2026[cel2026["Type"] == "leaf herbarium"] ##99

herb = pd.read_excel(r"C:\Users\selene.baez\Downloads\NAPO_preliminar\Jürgen herbarium\QCA_GOET.xlsx")
herb = herb[["numeroCampo","description"]]
herb = herb.drop_duplicates(subset="numeroCampo")
herb_2006["numeroCampo"] = herb_2006["numeroCampo"].astype("float64")
herb["numeroCampo"] = herb["numeroCampo"].astype("float64")

cel2026_herb = pd.merge(herb_2006,herb,left_on="numeroCampo",right_on="numeroCampo", how ="left")
cel2026_herb[["Plot_","TreeID_herb"]]= cel2026_herb["description"].str.split("#",expand=True)
cel2026_herb[["old TreeID","comment","comment2"]]= cel2026_herb["TreeID_herb"].str.split(",",expand=True)
cel2026_herb["Plot"]= cel2026_herb["Plot_"].str.replace("plot","").combine_first(cel2026_herb["Plot"])
cel2026_herb ### Plot that are JH look for them somewhere.
patrones = ["60", "62", "22","23","24","28","48","50","29"] #Galeras
herb_galeras = cel2026_herb[cel2026_herb["Plot"].str.contains("|".join(patrones), na=False)]
herb_galeras["old TreeID"] = herb_galeras["old TreeID"].astype("float64")
herb_galeras = pd.merge(herb_galeras, base_grande, left_on= "old TreeID", right_on = "treeID", how="inner")
herb_galeras["Site"] = "Galeras"

results_2026 = pd.concat([herb_galeras,pulv2026Complete])
#HELP THE CONTROLS ARE GIVING ME PROBLEMS

C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\1390077857.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pulv2026 [["Plot","new TreeID"]] = pulv2026["Sample-ID"].str.split("-",expand=True)
C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\1390077857.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pulv2026 [["Plot","new TreeID"]] = pulv2026["Sample-ID"].str.split("-",expand=True)
C:\Users\selene.baez\AppData\Local\Temp\ipykernel_13468\1390077857.py:5: SettingWithCopyWarning: 
A value 

# Nutrientes: C, N, delta15N

In [91]:

SEQ1= pd.read_excel("Andrea_bulk CN_2025.xlsx",sheet_name="SEQ1")
SEQ1["Year"] = 2006
SEQ2 = pd.read_excel("Andrea_bulk CN_2025.xlsx",sheet_name="SEQ2")
SEQ2["Year"] = 2006
SEQ4 = pd.read_excel("Andrea_bulk CN_2025.xlsx",sheet_name="SEQ4")
SEQ4["Year"] = 2025
SEQ5 = pd.read_excel("Andrea_bulk CN_2025.xlsx",sheet_name="SEQ5")
SEQ5["Year"] = 2025
SEQ7 = pd.read_excel("Andrea_bulk CN_2025.xlsx",sheet_name="SEQ7")
SEQ7["Year"] = 2006

bulk = pd.concat([SEQ1,SEQ2,SEQ4,SEQ5,SEQ7])
bulk = bulk.rename(columns={"certified values":"sample-ID", "Unnamed: 2":"[N]",
                            "Unnamed: 4":nameC_bulk,"Unnamed: 6":named15N,
                            "Unnamed: 8":named13C_bulk})


bulk[["Plot","TreeID","add"]] =bulk["sample-ID"].str.split("-",expand=True) #len 182
columns_drop = ["Unnamed: 0", "Unnamed: 3","Unnamed: 5","Unnamed: 7", "Unnamed: 9","Unnamed: 10","Unnamed: 11","Unnamed: 12"]
bulk = bulk.drop(columns=columns_drop)
patrones = ["p60", "p62", "p22","p23","p24","p28","p48","p50","p29"]
df_filtrado_bulk = bulk[bulk["sample-ID"].str.contains("|".join(patrones), na=False)]
df_filtrado_bulk = df_filtrado_bulk.drop(columns = {"Plot"})

In [92]:
merged_cellulosa_bulk = pd.merge(df_filtrado, df_filtrado_bulk,left_on="Sample ID", right_on="sample-ID",how= "outer")
len(merged_cellulosa_bulk) #82

82

In [93]:
merged_cellulosa_bulk["Year"] = merged_cellulosa_bulk["Year_x"].combine_first(
    merged_cellulosa_bulk["Year_y"]
)

merged_cellulosa_bulk["TreeID"] = merged_cellulosa_bulk["TreeID_x"].combine_first(
    merged_cellulosa_bulk["TreeID_y"]
)

merged_cellulosa_bulk["TreeID"] = merged_cellulosa_bulk["TreeID"].astype("Float64")


# Unir con la base de datos completa para añadir informacion adicional

In [96]:
### HELP
df_filtrado = merged_cellulosa_bulk
df_filtrado["Year"] = df_filtrado["Year"].astype("Float64")

df_2006 = df_filtrado[df_filtrado["Year"] == 2006.0]
print(len(df_2006)) #40
df_2025 = df_filtrado[df_filtrado["Year"] == 2025.0]
print(len(df_2025)) #42

### HELP: USAR BASE GRANDE O RASGOS. Base grande eliminar los x y y de los datos para evitar problemas. 
base_grade = base_grande[["treeID","new ID 2024","family","JH herbarium collections", "genus","species"]]
base_grande = base_grande[~base_grande["treeID"].isin(["x", "y"])]
base_grande["treeID"] = base_grande["treeID"].replace("", np.nan)
base_grande["treeID"] = base_grande["treeID"].astype("Float64")


merge_2006 = pd.merge(df_2006,base_grande, left_on = ["TreeID"], right_on = ["treeID"], how = "inner") #left. Prioridad a los isotopos 
merge_2006.to_excel("2006.xlsx")

base_grande["new ID 2024"] = base_grande["new ID 2024"].astype("Float64")
merge_2025 = pd.merge(df_2025,base_grande, left_on = ["TreeID"], right_on = ["new ID 2024"], how = "inner") #left. Prioridad a los isotopos 
merge_2025.to_excel("2025.xlsx")
###
Keep = ["Plot_x","family","genus","species","PlotID","treeID","new ID 2024",named13C_cellulose,"Year",
        "[N]","[C_b]%","δ15N (‰ v.s. V-PDB)","bulk_δ13C (‰ v.s.V-PDB)",
        "JH herbarium collections","Type","Sample ID"]

keep2 = ["Plot_x","family","genus","species","PlotID","treeID","new ID 2024",named13C_cellulose,"Year",
         "[C_b]%","δ15N (‰ v.s. V-PDB)","bulk_δ13C (‰ v.s.V-PDB)",
        "JH herbarium collections","Type","old TreeID","new TreeID","Sample-ID"]


merge_2006 = merge_2006[Keep]
merge_2025 = merge_2025[Keep]
results_2026= results_2026[keep2]
all = pd.concat([merge_2006,merge_2025,results_2026])
all = all.sort_values(by=["Plot_x","new ID 2024"])

all.to_excel("Gal.xlsx")

40
42


In [95]:
dups = all[all.duplicated(
    subset=["treeID", "new ID 2024"],
    keep=False
)]